# Week 6 — Object-Oriented Programming (OOP)
### Python for Blockchain Analytics | Phase 1

---

**What you'll learn this week:**
- What OOP is and *why* it exists
- Classes and objects — the blueprint and the instance
- `__init__` — initialising an object's state
- Instance methods — behaviour attached to an object
- Class attributes vs instance attributes
- Inheritance — building on existing classes
- Dunder (magic) methods — `__repr__`, `__str__`, `__eq__`, `__lt__`
- When to use OOP vs functions

**SQL analyst parallel:**
Think of a **class** as a table schema — it defines what columns every row has.
An **instance** (object) is one row in that table.
A **method** is a stored procedure that operates on one specific row.

```sql
-- This is like a class definition:
CREATE TABLE Token (
    symbol     VARCHAR,
    price_usd  DECIMAL,
    decimals   INT DEFAULT 18
);

-- This is like creating an instance:
INSERT INTO Token VALUES ('ETH', 3247.85, 18);
```

**Time:** ~3.5 hours

---

## 1. Why OOP Exists — The Problem It Solves

Before learning OOP, you need to understand the pain it was designed to fix.

Imagine you're tracking 50 tokens. Without OOP, you might use separate variables or dicts:

```python
eth_symbol   = "ETH"
eth_price    = 3247.85
eth_decimals = 18

btc_symbol   = "BTC"
btc_price    = 67412.0
btc_decimals = 8
```

Or a list of dicts:
```python
tokens = [
    {"symbol": "ETH", "price": 3247.85, "decimals": 18},
    {"symbol": "BTC", "price": 67412.0, "decimals": 8},
]
```

**What's wrong with this?**
- No validation — you can set `price = -500` and nothing stops you
- No behaviour — to compute market cap you write a separate function every time
- No guarantee of structure — what if someone forgets to add `"decimals"`?
- Hard to add methods that "belong" to a token (like converting amounts)

**OOP solves this** by bundling the data (attributes) AND the behaviour (methods)
into one unit called a **class**. The class guarantees structure and encapsulates logic.

In [ ]:
# Without OOP — fragile, repetitive
def get_token_value(token_dict, amount):
    return amount / 10**token_dict["decimals"] * token_dict["price"]

eth = {"symbol": "ETH", "price": 3247.85, "decimals": 18}
print(get_token_value(eth, 1_500_000_000_000_000_000))

# With OOP — clean, self-contained, validates its own data
# (full implementation coming in the next section)
# eth = Token("ETH", 3247.85, 18)
# print(eth.value_of(1_500_000_000_000_000_000))

print("""
OOP bundles together:
  DATA       → attributes  (what a thing HAS)
  BEHAVIOUR  → methods     (what a thing CAN DO)
""")

## 2. Classes and Objects — Blueprint vs Instance

A **class** is a blueprint. It describes what every object of that type will look like.
An **object** (also called an instance) is one specific thing created from that blueprint.

**Real-world analogy:**
- The Ethereum ERC-20 standard is the *class* — it defines what every token must have
- USDC, UNI, AAVE are *instances* — each one is a specific token following that standard

**Syntax:**
```python
class ClassName:
    def __init__(self, param1, param2):
        self.attribute1 = param1
        self.attribute2 = param2
```

**Explaining the terms**
1. The `class` keyword defines the blueprint.
2. `__init__` is called automatically when you create an instance — it sets up the object's initial state.
3. `self` refers to the specific instance being created or used — it's how the object refers to itself.

In [1]:
# Your first class — a simple Token

class Token:
    """Represents an ERC-20 token on a blockchain."""

    def __init__(self, symbol, price_usd, decimals=18):
        """
        Initialise a Token instance.

        self     → the specific Token object being created
        symbol   → stored as self.symbol on this object
        price_usd → stored as self.price_usd on this object
        decimals → stored as self.decimals (defaults to 18)
        """
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals

# Creating instances — each one is a separate object
eth  = Token("ETH",  3247.85, 18)
btc  = Token("BTC",  67412.0, 8)
usdc = Token("USDC", 1.00,    6)

# Accessing attributes with dot notation
print(eth.symbol)     # ETH
print(btc.price_usd)  # 67412.0
print(usdc.decimals)  # 6

# Each instance is independent — changing one doesn't affect others
eth.price_usd = 3310.0
print(f"ETH: ${eth.price_usd}")   # 3310.0
print(f"BTC: ${btc.price_usd}")   # 67412.0 — unchanged

# type() shows you what class an object belongs to
print(type(eth))          # <class '__main__.Token'>
print(isinstance(eth, Token))  # True

ETH
67412.0
6
ETH: $3310.0
BTC: $67412.0
<class '__main__.Token'>
True


## 3. Instance Methods — What an Object Can Do

A **method** is a function defined inside a class.
The key difference from a regular function: the first parameter is always `self`,
which gives the method access to the object's own attributes.

When you call `eth.value_of(1_500_000_000_000_000_000)`, Python automatically
passes `eth` as the `self` argument — you never pass it explicitly.

Think of methods as asking the object to do something for you:
- `eth.value_of(amount)` — "Hey ETH token, what is this raw amount worth?"
- `wallet.deposit(eth, 1.5)` — "Hey wallet, deposit 1.5 ETH into yourself"

This is the core idea of OOP: **objects know how to do things with their own data**.

In [2]:
class Token:
    """
    Represents an ERC-20 token.

    Attributes:
        symbol    (str):   Token ticker symbol e.g. "ETH"
        price_usd (float): Current price in USD
        decimals  (int):   Token decimal places (default 18)
    """

    def __init__(self, symbol: str, price_usd: float, decimals: int = 18):
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals

    # ── Conversion methods ────────────────────────────────────
    def to_human(self, raw_amount: int) -> float:
        """Convert a raw on-chain integer to a human-readable float.

        On-chain token amounts are stored as integers scaled by 10^decimals.
        For ETH (18 decimals): 1 ETH = 1_000_000_000_000_000_000 (1e18)
        For USDC (6 decimals): 1 USDC = 1_000_000
        """
        return raw_amount / 10**self.decimals

    def to_raw(self, human_amount: float) -> int:
        """Convert a human-readable amount back to the raw on-chain integer."""
        return int(human_amount * 10**self.decimals)

    def value_of(self, raw_amount: int) -> float:
        """Return the USD value of a raw token amount.

        Example: eth.value_of(1_500_000_000_000_000_000)
        First converts to human (1.5 ETH), then multiplies by price.
        """
        return self.to_human(raw_amount) * self.price_usd

    # ── Price methods ─────────────────────────────────────────
    def update_price(self, new_price: float) -> None:
        """Update the token's price. Raises ValueError for negative prices."""
        if new_price < 0:
            raise ValueError(f"Price cannot be negative, got {new_price}")
        self.price_usd = new_price

    def price_change_pct(self, old_price: float) -> float:
        """Calculate % price change from old_price to current price."""
        if old_price == 0:
            raise ValueError("Old price cannot be zero")
        return (self.price_usd - old_price) / old_price * 100

    # ── Info methods ──────────────────────────────────────────
    def is_stablecoin(self) -> bool:
        """Return True if this token is likely a stablecoin (price within 1% of $1)."""
        return abs(self.price_usd - 1.0) < 0.01

    def format_amount(self, raw_amount: int, precision: int = 4) -> str:
        """Format a raw amount as a human-readable string with symbol."""
        human = self.to_human(raw_amount)
        return f"{human:,.{precision}f} {self.symbol}"

# ── Using the Token class ─────────────────────────────────────
eth  = Token("ETH",  3247.85, 18)
usdc = Token("USDC", 1.0003,  6)

# Conversion
raw_eth = 1_500_000_000_000_000_000   # 1.5 ETH in Wei
print(f"Raw: {raw_eth}")
print(f"Human: {eth.to_human(raw_eth)} ETH")
print(f"Value: ${eth.value_of(raw_eth):,.2f}")
print(f"Formatted: {eth.format_amount(raw_eth)}")

# Round trip: human → raw → human
human_amount = 2.5
raw = eth.to_raw(human_amount)
back = eth.to_human(raw)
print(f"\nRound trip: {human_amount} ETH → {raw:,} Wei → {back} ETH")

# Price update
old = eth.price_usd
eth.update_price(3310.0)
print(f"\nPrice change: {eth.price_change_pct(old):+.2f}%")

# Stablecoin check
print(f"\nETH is stablecoin: {eth.is_stablecoin()}")
print(f"USDC is stablecoin: {usdc.is_stablecoin()}")

# Error handling
try:
    eth.update_price(-100)
except ValueError as e:
    print(f"\nCaught error: {e}")

Raw: 1500000000000000000
Human: 1.5 ETH
Value: $4,871.77
Formatted: 1.5000 ETH

Round trip: 2.5 ETH → 2,500,000,000,000,000,000 Wei → 2.5 ETH

Price change: +1.91%

ETH is stablecoin: False
USDC is stablecoin: True

Caught error: Price cannot be negative, got -100


## 4. Class Attributes vs Instance Attributes

There are two places to store data in a class:

**Instance attributes** (`self.something`) — unique to each object.
Every `Token` instance has its own `symbol`, `price_usd`, and `decimals`.

**Class attributes** (defined directly in the class body) — shared across ALL instances.
If you store a counter of how many tokens have been created, that belongs to the class, not any single token.

**When to use each:**
- Instance attribute → data that differs between objects (price, symbol, balance)
- Class attribute    → data shared by all objects (constants, counters, shared config)

In [3]:
class Token:
    """Token with both class and instance attributes."""

    # ── Class attributes — shared by ALL Token instances ──────
    KNOWN_STABLECOINS = {"USDC", "USDT", "DAI", "FRAX", "LUSD"}
    _instance_count   = 0       # tracks how many Token objects exist

    def __init__(self, symbol: str, price_usd: float, decimals: int = 18):
        # ── Instance attributes — unique to each Token ────────
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals

        # Update the class-level counter every time a new Token is created
        Token._instance_count += 1
        self.id = Token._instance_count   # each token gets a unique ID

    def is_stablecoin(self) -> bool:
        """Check against the class-level known stablecoins set."""
        return self.symbol in Token.KNOWN_STABLECOINS

    @classmethod
    def get_count(cls) -> int:
        """Return total number of Token instances created.
        @classmethod receives the CLASS as first arg (cls), not an instance (self).
        Use it for operations that relate to the class itself, not one instance.
        """
        return cls._instance_count

    @staticmethod
    def is_valid_symbol(symbol: str) -> bool:
        """Check if a string is a valid token symbol (1-10 uppercase letters).
        @staticmethod doesn't receive self or cls — it's just a utility function
        that logically belongs here but doesn't need any object or class data.
        """
        return (
            isinstance(symbol, str)
            and 1 <= len(symbol) <= 10
            and symbol.isalpha()
            and symbol.isupper()
        )

# Class attributes are accessed on the class itself
print("Known stablecoins:", Token.KNOWN_STABLECOINS)

# Create some instances
eth  = Token("ETH",  3247.85)
btc  = Token("BTC",  67412.0, decimals=8)
usdc = Token("USDC", 1.00, decimals=6)

print(f"\nETH id: {eth.id}")
print(f"BTC id: {btc.id}")
print(f"USDC id: {usdc.id}")

# classmethod — called on the class
print(f"\nTotal tokens created: {Token.get_count()}")

# staticmethod — called on the class (no self or cls)
for sym in ["ETH", "eth", "ETH1", "TOOLONGSYMBOL", "UNI"]:
    print(f"  '{sym}' valid symbol: {Token.is_valid_symbol(sym)}")

# Instance method — called on an object
print(f"\nETH is stablecoin: {eth.is_stablecoin()}")
print(f"USDC is stablecoin: {usdc.is_stablecoin()}")

Known stablecoins: {'FRAX', 'USDC', 'USDT', 'LUSD', 'DAI'}

ETH id: 1
BTC id: 2
USDC id: 3

Total tokens created: 3
  'ETH' valid symbol: True
  'eth' valid symbol: False
  'ETH1' valid symbol: False
  'TOOLONGSYMBOL' valid symbol: False
  'UNI' valid symbol: True

ETH is stablecoin: False
USDC is stablecoin: True


## 5. Dunder (Magic) Methods

**Dunder** stands for "double underscore" — methods like `__repr__`, `__str__`, `__eq__`.

These methods let your objects work with Python's built-in syntax:
- `print(token)` calls `__str__`
- `repr(token)` calls `__repr__`
- `token_a == token_b` calls `__eq__`
- `token_a < token_b` calls `__lt__`
- `len(wallet)` calls `__len__`
- `token in wallet` calls `__contains__`

They make your custom objects feel like native Python types.

**`__repr__` vs `__str__`:**
- `__repr__` — unambiguous, for developers. Should ideally be code that recreates the object.
- `__str__` — readable, for users/display. What you see when you `print()`.

If you only define one, define `__repr__` — Python falls back to it for both.

In [4]:
class Token:
    """Token with dunder methods."""

    KNOWN_STABLECOINS = {"USDC", "USDT", "DAI", "FRAX"}

    def __init__(self, symbol: str, price_usd: float, decimals: int = 18):
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals

    def __repr__(self) -> str:
        """
        Unambiguous developer representation.
        Convention: return a string that looks like the constructor call.
        Shown in the REPL, in lists, in debuggers.
        """
        return f"Token(symbol={self.symbol!r}, price_usd={self.price_usd}, decimals={self.decimals})"

    def __str__(self) -> str:
        """
        Readable user-facing representation.
        Used by print() and str().
        """
        category = "stablecoin" if self.symbol in self.KNOWN_STABLECOINS else "token"
        return f"{self.symbol} {category} @ ${self.price_usd:,.2f}"

    def __eq__(self, other) -> bool:
        """
        Define what == means for two Token objects.
        Two tokens are equal if they have the same symbol (case-insensitive).
        We also handle the case where other might not be a Token.
        """
        if not isinstance(other, Token):
            return NotImplemented
        return self.symbol.upper() == other.symbol.upper()

    def __lt__(self, other) -> bool:
        """
        Define what < means — allows sorted() to work on a list of Tokens.
        We sort by price ascending.
        """
        if not isinstance(other, Token):
            return NotImplemented
        return self.price_usd < other.price_usd

    def __len__(self) -> int:
        """
        len(token) returns the length of the symbol string.
        A creative but valid use of __len__.
        """
        return len(self.symbol)

    def __contains__(self, item) -> bool:
        """
        Allows: 'E' in eth_token  →  checks if char is in symbol
        """
        return item in self.symbol

# ── Dunder methods in action ───────────────────────────────
eth  = Token("ETH",  3247.85)
btc  = Token("BTC",  67412.0, 8)
usdc = Token("USDC", 1.00, 6)
uni  = Token("UNI",  12.84)

# __repr__
print("repr:", repr(eth))

# __str__
print("str: ", str(eth))
print("print:", eth)          # print() calls __str__

# __eq__
eth2 = Token("ETH", 3310.0)   # different price, same symbol
print(f"\neth == eth2: {eth == eth2}")    # True — same symbol
print(f"eth == btc:  {eth == btc}")      # False
print(f"eth == 'ETH': {eth == 'ETH'}")   # NotImplemented → False

# __lt__ — now sorted() works!
tokens = [btc, eth, usdc, uni]
sorted_tokens = sorted(tokens)            # uses __lt__ (sort by price)
print("\nSorted by price:")
for t in sorted_tokens:
    print(f"  {t}")                       # uses __str__

# __len__
print(f"\nlen(eth): {len(eth)}")         # 3
print(f"len(usdc): {len(usdc)}")         # 4

# __contains__
print(f"\n'E' in eth: {'E' in eth}")     # True
print(f"'X' in eth: {'X' in eth}")       # False

# Tokens in a list
print(f"\neth in [eth, btc]: {eth in [eth, btc]}")  # True (uses __eq__)

repr: Token(symbol='ETH', price_usd=3247.85, decimals=18)
str:  ETH token @ $3,247.85
print: ETH token @ $3,247.85

eth == eth2: True
eth == btc:  False
eth == 'ETH': False

Sorted by price:
  USDC stablecoin @ $1.00
  UNI token @ $12.84
  ETH token @ $3,247.85
  BTC token @ $67,412.00

len(eth): 3
len(usdc): 4

'E' in eth: True
'X' in eth: False

eth in [eth, btc]: True


## 6. Inheritance — Building on Existing Classes

**Inheritance** lets you create a new class that builds on an existing one.

The new class (child/subclass) **inherits** all attributes and methods of the
parent class, and can:
- Add new attributes and methods
- Override (replace) parent methods with specialised behaviour
- Call the parent's version of a method with `super()`

**Blockchain analogy:**
The ERC-20 standard is the parent. Every token (USDC, UNI, AAVE) is a child
that inherits the standard interface and adds its own specifics on top.

```
Token (parent)
├── ERC20Token (child) — adds supply, allowances
│   ├── StableToken (grandchild) — adds peg tracking
│   └── GovernanceToken (grandchild) — adds voting power
└── NFT (child) — completely different behaviour
```

In [1]:
class Token:
    """Base Token class — the parent."""

    def __init__(self, symbol: str, price_usd: float, decimals: int = 18):
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals

    def to_human(self, raw: int) -> float:
        return raw / 10**self.decimals

    def value_of(self, raw: int) -> float:
        return self.to_human(raw) * self.price_usd

    def __repr__(self):
        return f"Token({self.symbol!r}, ${self.price_usd})"

    def __str__(self):
        return f"{self.symbol} @ ${self.price_usd:,.2f}"

    def info(self) -> str:
        """Return a basic description of the token."""
        return f"{self.symbol}: ${self.price_usd:,.2f} | {self.decimals} decimals"


# ── ERC20Token inherits from Token ───────────────────────────
class ERC20Token(Token):
    """
    An ERC-20 token with total supply and contract address.
    Inherits everything from Token and adds ERC-20 specific data.
    """

    def __init__(self, symbol: str, price_usd: float,
                 total_supply: int, contract_address: str,
                 decimals: int = 18):
        # super() calls the parent's __init__ — sets symbol, price_usd, decimals
        # Always call super().__init__() first when extending a parent
        super().__init__(symbol, price_usd, decimals)

        # Now add ERC-20 specific attributes
        self.total_supply       = total_supply
        self.contract_address   = contract_address

    def market_cap(self) -> float:
        """Market cap = circulating supply * price.
        Uses parent's to_human() and self's price_usd — no duplication.
        """
        return self.to_human(self.total_supply) * self.price_usd

    def short_address(self) -> str:
        """Return shortened contract address."""
        a = self.contract_address
        return f"{a[:6]}...{a[-4:]}"

    def info(self) -> str:
        """Override parent's info() to include supply and market cap."""
        base = super().info()   # get parent's version first
        mcap = self.market_cap()
        return f"{base} | Supply: {self.to_human(self.total_supply):,.0f} | MCap: ${mcap/1e9:.2f}B"


# ── StableToken inherits from ERC20Token ─────────────────────
class StableToken(ERC20Token):
    """
    A stablecoin — an ERC-20 token pegged to a fiat currency.
    Adds peg tracking and deviation alerts.
    """

    def __init__(self, symbol: str, price_usd: float,
                 total_supply: int, contract_address: str,
                 peg_target: float = 1.00, decimals: int = 6):
        # Stablecoins usually have 6 decimals (USDC, USDT)
        super().__init__(symbol, price_usd, total_supply, contract_address, decimals)
        self.peg_target = peg_target

    def peg_deviation(self) -> float:
        """Return how far the price is from the peg target (%)."""
        return (self.price_usd - self.peg_target) / self.peg_target * 100

    def is_depegged(self, threshold_pct: float = 0.5) -> bool:
        """Return True if price deviates from peg by more than threshold."""
        return abs(self.peg_deviation()) > threshold_pct

    def info(self) -> str:
        dev  = self.peg_deviation()
        flag = "⚠️  DEPEGGED" if self.is_depegged() else "✅ On peg"
        return f"{super().info()} | Peg dev: {dev:+.4f}% {flag}"


# ── Using the hierarchy ────────────────────────────────────────
eth_base = Token("ETH", 3247.85)

uni = ERC20Token(
    symbol           = "UNI",
    price_usd        = 12.84,
    total_supply     = 1_000_000_000 * 10**18,
    contract_address = "0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984",
)

usdc = StableToken(
    symbol           = "USDC",
    price_usd        = 0.9987,    # slightly off-peg
    total_supply     = 43_000_000_000 * 10**6,
    contract_address = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48",
)

usdt = StableToken(
    symbol           = "USDT",
    price_usd        = 1.0001,
    total_supply     = 110_000_000_000 * 10**6,
    contract_address = "0xdAC17F958D2ee523a2206206994597C13D831ec7",
)

# All three share Token's interface
print("=== Token hierarchy ===")
for token in [eth_base, uni, usdc, usdt]:
    print(token.info())
    print()

# isinstance checks — a child IS an instance of all its ancestors
print("isinstance checks:")
print(f"usdc isinstance StableToken: {isinstance(usdc, StableToken)}")  # True
print(f"usdc isinstance ERC20Token:  {isinstance(usdc, ERC20Token)}")   # True
print(f"usdc isinstance Token:       {isinstance(usdc, Token)}")         # True
print(f"uni  isinstance StableToken: {isinstance(uni, StableToken)}")    # False

# Inherited methods work on child classes
print(f"\nUNI market cap: ${uni.market_cap()/1e9:.2f}B")
print(f"USDC market cap: ${usdc.market_cap()/1e9:.2f}B")
print(f"\nUSDC depegged: {usdc.is_depegged()}")    # True (0.13% off)
print(f"USDT depegged: {usdt.is_depegged()}")       # False

=== Token hierarchy ===
ETH: $3,247.85 | 18 decimals

UNI: $12.84 | 18 decimals | Supply: 1,000,000,000 | MCap: $12.84B

USDC: $1.00 | 6 decimals | Supply: 43,000,000,000 | MCap: $42.94B | Peg dev: -0.1300% ✅ On peg

USDT: $1.00 | 6 decimals | Supply: 110,000,000,000 | MCap: $110.01B | Peg dev: +0.0100% ✅ On peg

isinstance checks:
usdc isinstance StableToken: True
usdc isinstance ERC20Token:  True
usdc isinstance Token:       True
uni  isinstance StableToken: False

UNI market cap: $12.84B
USDC market cap: $42.94B

USDC depegged: False
USDT depegged: False


## 7. A Complete Real-World Example — The Wallet Class

Now let's build something substantial: a `Wallet` class that models an
Ethereum wallet with token holdings, transaction history, and analytics.

This is the kind of class you'd actually write in production blockchain code.
Notice how it uses everything from this week AND from previous weeks:
- `__init__`, methods, dunder methods (Weeks 5-6)
- Lists, dicts, sets (Week 4)
- Control structures (Week 3)
- F-strings, type conversions (Week 2)

In [2]:
from collections import defaultdict

class Token:
    """Minimal Token for use inside Wallet."""
    def __init__(self, symbol, price_usd, decimals=18):
        self.symbol    = symbol
        self.price_usd = price_usd
        self.decimals  = decimals
    def to_human(self, raw):
        return raw / 10**self.decimals
    def __repr__(self):
        return f"Token({self.symbol!r})"


class Wallet:
    """
    Models an Ethereum wallet with token holdings and transaction history.

    Attributes:
        address    (str):  The 0x wallet address
        label      (str):  Optional human-readable label (e.g. "Vitalik")
        holdings   (dict): {token_symbol → human_amount}
        tx_history (list): List of transaction dicts
        _protocols (set):  Protocols this wallet has interacted with
    """

    def __init__(self, address: str, label: str = ""):
        # Validate address before accepting it
        if not address.startswith("0x") or len(address) != 42:
            raise ValueError(f"Invalid Ethereum address: {address}")

        self.address    = address
        self.label      = label or self._shorten(address)
        self.holdings   = {}          # symbol → balance (in human-readable units)
        self.tx_history = []
        self._protocols = set()       # private: protocols interacted with

    # ── Private helpers (convention: prefix with _) ───────────
    def _shorten(self, address: str) -> str:
        """Shorten an address for display (private helper)."""
        return f"{address[:6]}...{address[-4:]}"

    def _record_tx(self, tx_type: str, **details) -> dict:
        """Record a transaction in history and return it."""
        import time
        tx = {"type": tx_type, "timestamp": int(time.time()), **details}
        self.tx_history.append(tx)
        return tx

    # ── Core operations ───────────────────────────────────────
    def deposit(self, token: Token, amount: float) -> None:
        """
        Deposit tokens into the wallet.

        Args:
            token:  Token object to deposit
            amount: Amount in human-readable units (e.g. 1.5 for 1.5 ETH)
        """
        if amount <= 0:
            raise ValueError(f"Deposit amount must be positive, got {amount}")

        symbol = token.symbol
        self.holdings[symbol] = self.holdings.get(symbol, 0.0) + amount
        self._record_tx("deposit", symbol=symbol, amount=amount,
                        price_usd=token.price_usd,
                        value_usd=amount * token.price_usd)
        print(f"  ✅ Deposited {amount} {symbol} → {self.label}")

    def withdraw(self, token: Token, amount: float) -> None:
        """
        Withdraw tokens from the wallet.

        Raises InsufficientFundsError if balance is too low.
        """
        if amount <= 0:
            raise ValueError(f"Withdrawal amount must be positive, got {amount}")

        symbol  = token.symbol
        balance = self.holdings.get(symbol, 0.0)

        if balance < amount:
            raise ValueError(
                f"Insufficient {symbol}: have {balance:.4f}, need {amount:.4f}"
            )

        self.holdings[symbol] -= amount
        if self.holdings[symbol] == 0:
            del self.holdings[symbol]   # clean up zero balances

        self._record_tx("withdraw", symbol=symbol, amount=amount,
                        price_usd=token.price_usd,
                        value_usd=amount * token.price_usd)
        print(f"  ✅ Withdrew {amount} {symbol} from {self.label}")

    def interact_with(self, protocol: str) -> None:
        """Record a protocol interaction (e.g. Uniswap, Aave)."""
        self._protocols.add(protocol)

    # ── Analytics ─────────────────────────────────────────────
    def total_value_usd(self, prices: dict) -> float:
        """
        Calculate total portfolio value in USD.

        Args:
            prices: {symbol → current_price_usd} — pass fresh prices from an API

        Returns:
            float: Total portfolio value in USD
        """
        total = 0.0
        for symbol, amount in self.holdings.items():
            price = prices.get(symbol, 0)
            total += amount * price
        return total

    def portfolio_breakdown(self, prices: dict) -> list:
        """
        Return sorted list of holdings with USD values and allocations.

        Returns:
            list of dicts sorted by USD value descending
        """
        total = self.total_value_usd(prices)
        rows = []
        for symbol, amount in self.holdings.items():
            price     = prices.get(symbol, 0)
            value_usd = amount * price
            alloc_pct = (value_usd / total * 100) if total > 0 else 0
            rows.append({
                "symbol":    symbol,
                "amount":    amount,
                "price":     price,
                "value_usd": value_usd,
                "alloc_pct": alloc_pct,
            })
        return sorted(rows, key=lambda r: -r["value_usd"])

    def tx_count(self, tx_type: str = None) -> int:
        """Count transactions, optionally filtered by type."""
        if tx_type is None:
            return len(self.tx_history)
        return sum(1 for tx in self.tx_history if tx["type"] == tx_type)

    def classify(self) -> str:
        """Classify the wallet based on its activity."""
        n_txns = len(self.tx_history)
        n_protocols = len(self._protocols)

        if n_txns > 500:              return "🤖 Bot / High-frequency"
        if n_protocols >= 5:          return "⚡ DeFi Power User"
        if n_protocols >= 2:          return "👤 DeFi User"
        if n_txns > 20:               return "📊 Active Retail"
        return "🐣 New User"

    # ── Dunder methods ────────────────────────────────────────
    def __repr__(self) -> str:
        return f"Wallet(address={self.address!r}, holdings={list(self.holdings.keys())})"

    def __str__(self) -> str:
        return f"[{self.label}] {len(self.holdings)} tokens | {len(self.tx_history)} txns"

    def __len__(self) -> int:
        """len(wallet) = number of different tokens held."""
        return len(self.holdings)

    def __contains__(self, symbol: str) -> bool:
        """'ETH' in wallet → True if wallet holds ETH."""
        return symbol in self.holdings

    def __eq__(self, other) -> bool:
        """Two wallets are equal if they share the same address."""
        if not isinstance(other, Wallet):
            return NotImplemented
        return self.address.lower() == other.address.lower()

# ── Demo ──────────────────────────────────────────────────────
eth  = Token("ETH",  3247.85, 18)
btc  = Token("BTC",  67412.0, 8)
usdc = Token("USDC", 1.00,    6)
uni  = Token("UNI",  12.84,   18)

wallet = Wallet("0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045", label="Vitalik")

print("── Deposits ──")
wallet.deposit(eth,  3.5)
wallet.deposit(btc,  0.1)
wallet.deposit(usdc, 5_000)
wallet.deposit(uni,  500)
wallet.deposit(eth,  1.0)   # additional deposit

print("\n── Withdrawal ──")
wallet.withdraw(usdc, 1_000)

# Protocol interactions
wallet.interact_with("Uniswap V3")
wallet.interact_with("Aave V3")
wallet.interact_with("Curve")

# Dunder methods
print(f"\nrepr: {repr(wallet)}")
print(f"str:  {wallet}")
print(f"len:  {len(wallet)} tokens")
print(f"'ETH' in wallet: {'ETH' in wallet}")
print(f"'SOL' in wallet: {'SOL' in wallet}")

# Analytics
CURRENT_PRICES = {"ETH": 3247.85, "BTC": 67412.0, "USDC": 1.00, "UNI": 12.84}

print(f"\n── Portfolio ──")
print(f"Total value: ${wallet.total_value_usd(CURRENT_PRICES):,.2f}")
print(f"Classification: {wallet.classify()}")
print(f"\nBreakdown:")
print(f"  {'Symbol':<6}  {'Amount':>10}  {'Price':>10}  {'Value':>12}  {'Alloc%':>7}")
print("  " + "-" * 50)
for row in wallet.portfolio_breakdown(CURRENT_PRICES):
    print(f"  {row['symbol']:<6}  {row['amount']:>10,.4f}  "
          f"${row['price']:>9,.2f}  ${row['value_usd']:>11,.2f}  "
          f"{row['alloc_pct']:>6.1f}%")

── Deposits ──
  ✅ Deposited 3.5 ETH → Vitalik
  ✅ Deposited 0.1 BTC → Vitalik
  ✅ Deposited 5000 USDC → Vitalik
  ✅ Deposited 500 UNI → Vitalik
  ✅ Deposited 1.0 ETH → Vitalik

── Withdrawal ──
  ✅ Withdrew 1000 USDC from Vitalik

repr: Wallet(address='0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045', holdings=['ETH', 'BTC', 'USDC', 'UNI'])
str:  [Vitalik] 4 tokens | 6 txns
len:  4 tokens
'ETH' in wallet: True
'SOL' in wallet: False

── Portfolio ──
Total value: $31,776.53
Classification: 👤 DeFi User

Breakdown:
  Symbol      Amount       Price         Value   Alloc%
  --------------------------------------------------
  ETH         4.5000  $ 3,247.85  $  14,615.32    46.0%
  BTC         0.1000  $67,412.00  $   6,741.20    21.2%
  UNI       500.0000  $    12.84  $   6,420.00    20.2%
  USDC    4,000.0000  $     1.00  $   4,000.00    12.6%


## 8. When to Use OOP vs Functions

OOP is a tool, not a religion. Not everything needs to be a class.

**Use a class when:**
- You have data + behaviour that belong together (Token has price AND can convert)
- You need multiple instances with shared structure (100 wallets, 50 tokens)
- You need inheritance — different types share a base interface (Token → ERC20Token → StableToken)
- State changes over time — a Wallet's balance changes, and you want methods to manage that safely

**Use functions when:**
- You have one-off transformations (convert Wei to ETH — just `def wei_to_eth(w): return w/1e18`)
- No persistent state needed
- Simple utilities that don't belong to any particular object

**A common mistake is over-engineering:**
```python
# ❌ Don't do this — a class with no real state or behaviour
class WeiConverter:
    def convert(self, wei):
        return wei / 10**18

# ✅ Just use a function
def wei_to_eth(wei):
    return wei / 10**18
```

For this course, the rule is: if you're modelling a real-world entity
(Token, Wallet, Transaction, Pool, Protocol) that has both data and operations — use a class.

In [3]:
# Quick decision guide in code

# ── Use a function ────────────────────────────────────────────
# Pure transformation — input in, output out, no state
def wei_to_eth(wei: int) -> float:
    return wei / 10**18

def shorten_address(address: str) -> str:
    return f"{address[:6]}...{address[-4:]}"

def gas_cost_usd(gas_used: int, gwei: float, eth_price: float) -> float:
    return gas_used * gwei * 1e9 / 1e18 * eth_price

# ── Use a class ───────────────────────────────────────────────
# Entity with state that evolves over time
class Transaction:
    """
    Models an Ethereum transaction — data + derived properties in one place.
    """
    def __init__(self, tx_hash, from_addr, to_addr, value_wei,
                 gas_used, gas_price_gwei, status="success"):
        self.tx_hash        = tx_hash
        self.from_addr      = from_addr
        self.to_addr        = to_addr
        self.value_wei      = value_wei
        self.gas_used       = gas_used
        self.gas_price_gwei = gas_price_gwei
        self.status         = status

    @property
    def value_eth(self) -> float:
        """
        @property lets you access this like an attribute (tx.value_eth)
        instead of a method call (tx.value_eth()).
        The value is computed from self.value_wei each time — always fresh.
        """
        return self.value_wei / 10**18

    @property
    def gas_cost_eth(self) -> float:
        return self.gas_used * self.gas_price_gwei * 1e9 / 1e18

    @property
    def is_success(self) -> bool:
        return self.status == "success"

    def receipt(self, eth_price_usd: float = 3247.85) -> str:
        """Generate a formatted transaction receipt."""
        lines = [
            "=" * 50,
            "  TRANSACTION RECEIPT",
            "=" * 50,
            f"  Hash   : {self.tx_hash[:18]}...",
            f"  From   : {shorten_address(self.from_addr)}",
            f"  To     : {shorten_address(self.to_addr)}",
            f"  Status : {'✅ Success' if self.is_success else '❌ Failed'}",
            "-" * 50,
            f"  Value  : {self.value_eth:.6f} ETH  (${self.value_eth * eth_price_usd:,.2f})",
            f"  Gas    : {self.gas_used:,} units @ {self.gas_price_gwei} Gwei",
            f"  Fee    : {self.gas_cost_eth:.8f} ETH  (${self.gas_cost_eth * eth_price_usd:.4f})",
            "=" * 50,
        ]
        return "\n".join(lines)

    def __repr__(self):
        return f"Transaction({self.tx_hash[:10]}..., {self.value_eth:.4f} ETH, {self.status})"

# Using @property — no parentheses needed
tx = Transaction(
    tx_hash        = "0x5c504ed432cb51138bcf09aa5e8a410dd4a1e204ef84bfed1be16dfba1b22060",
    from_addr      = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    to_addr        = "0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984",
    value_wei      = 1_500_000_000_000_000_000,
    gas_used       = 21_000,
    gas_price_gwei = 20,
)

# @property accessed like attributes, not method calls
print(f"value_eth: {tx.value_eth}")        # no ()
print(f"gas_cost:  {tx.gas_cost_eth:.8f}") # no ()
print(f"success:   {tx.is_success}")       # no ()
print()
print(tx.receipt())

value_eth: 1.5
gas_cost:  0.00042000
success:   True

  TRANSACTION RECEIPT
  Hash   : 0x5c504ed432cb5113...
  From   : 0xd8dA...6045
  To     : 0x1f98...F984
  Status : ✅ Success
--------------------------------------------------
  Value  : 1.500000 ETH  ($4,871.77)
  Gas    : 21,000 units @ 20 Gwei
  Fee    : 0.00042000 ETH  ($1.3641)


## Summary

| Concept | What it is | Blockchain use |
|---------|-----------|----------------|
| `class` | Blueprint for objects | Token, Wallet, Transaction, Pool |
| `__init__` | Sets up object's initial state | Store symbol, price, address |
| Instance method | Function that uses `self` | `.deposit()`, `.value_of()`, `.classify()` |
| Class attribute | Shared across all instances | `KNOWN_STABLECOINS`, `_instance_count` |
| `@classmethod` | Method that receives the class | Factory methods, counters |
| `@staticmethod` | Utility that belongs in the class | `is_valid_symbol()` |
| `@property` | Computed attribute (no `()`) | `value_eth`, `gas_cost_eth` |
| `__repr__` | Developer-readable string | Shown in REPL and debugger |
| `__str__` | User-readable string | Used by `print()` |
| `__eq__` | Defines `==` between objects | Compare tokens by symbol |
| `__lt__` | Defines `<` — enables `sorted()` | Sort tokens by price |
| Inheritance | Child class extends parent | StableToken extends ERC20Token extends Token |
| `super()` | Call parent's method | Extend `__init__` without rewriting it |

---

## Phase 1 Complete 🎉

You've finished all 6 weeks of Python Fundamentals. Here's what you can now do:

- ✅ Write Python from scratch — syntax, data types, control flow
- ✅ Work with all four data structures — lists, tuples, dicts, sets
- ✅ Write reusable functions with proper signatures and docstrings
- ✅ Build a module of utilities (`blockchain_utils.py`)
- ✅ Model real-world blockchain entities as classes (Token, Wallet, Transaction)
- ✅ Use inheritance to build hierarchies (Token → ERC20Token → StableToken)

**Next: Phase 2 — Python for Data Analysis**
Week 7 kicks off with file handling and error handling, then Pandas, APIs, and visualization.

---

**Your task before Week 7:**
1. Complete `exercises.py` — build the `Transaction` class from scratch
2. Add `__hash__` to your Token class so tokens can be used in sets
3. Commit everything:
```bash
git add phase-1-python-fundamentals/week-06-oop/
git commit -m "phase-1/week-06: OOP — lesson, exercises, capstone"
git push
```